In [8]:
import numpy as np
import pandas as pd
import torch.nn.functional as F
import timm

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as T


class MVTecAD(Dataset):
    """
    Structure attendue :
    data/mv_tec_ad/
        <category>/
            train/
                good/*.png
            test/
                good/*.png
                <anomaly_type_1>/*.png
                <anomaly_type_2>/*.png
                ...
            ground_truth/
                <anomaly_type_1>/*_mask.png
                ...
    """

    def __init__(self, root="data/mv_tec_ad", category="bottle", split="train", transform=None):
        self.root = Path(root) / category
        self.split = split
        self.transform = transform or T.Compose([
            T.Resize((256, 256)),
            T.ToTensor(),
        ])
        self.samples = []  # (image_path, label, defect_type, mask_path)

        if split == "train":
            good_dir = self.root / "train" / "good"
            for img_path in sorted(good_dir.glob("*.png")):
                # label=0 (normal), pas de type d'anomalie, pas de masque
                self.samples.append((img_path, 0, "good", None))

        elif split == "test":
            test_dir = self.root / "test"
            for defect_dir in sorted(test_dir.iterdir()):
                if not defect_dir.is_dir():
                    continue

                defect_type = defect_dir.name          # ex: "good", "broken_large", "scratch"...
                label = 0 if defect_type == "good" else 1

                for img_path in sorted(defect_dir.glob("*.png")):
                    mask_path = None
                    if label == 1:
                        candidate = (
                            self.root / "ground_truth" / defect_type /
                            f"{img_path.stem}_mask.png"
                        )
                        mask_path = candidate if candidate.exists() else None

                    self.samples.append((img_path, label, defect_type, mask_path))

        else:
            raise ValueError(f"split doit être 'train' ou 'test', reçu : {split}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label, defect_type, mask_path = self.samples[idx]

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        if mask_path is not None:
            mask = Image.open(mask_path).convert("L")
            mask = T.Resize((256, 256))(mask)
            mask = T.ToTensor()(mask)
        else:
            # masque vide pour les images normales / sans ground truth
            mask = torch.zeros((1, 256, 256))

        return {
            "image": image,
            "label": label,
            "defect_type": defect_type,
            "mask": mask,
            "path": str(img_path),
        }

In [3]:
base_dir = "FLOCAT-GENAGN/What-if-Flocat-were-truly-generic-and-agnostic/data/mv_tec_ad"
train_dataset = MVTecAD(root=base_dir, category="bottle", split="train")
test_dataset = MVTecAD(root=base_dir, category="bottle", split="test")

print(f"Train (normal only) : {len(train_dataset)} images")
print(f"Test (normal + anomalies) : {len(test_dataset)} images")

sample = test_dataset[0]
print(sample["label"], sample["defect_type"], sample["image"].shape)

from collections import Counter
defect_counts = Counter(s[2] for s in test_dataset.samples)
print(defect_counts)

Train (normal only) : 209 images
Test (normal + anomalies) : 83 images
1 broken_large torch.Size([3, 256, 256])
Counter({'broken_small': 22, 'contamination': 21, 'broken_large': 20, 'good': 20})


In [ ]:
def embed_patches(features, patch_size=3):
    """
    Agrège l'info locale autour de chaque position (voisinage p=3, stride=1)
    via un average pooling, comme dans PatchCore.
    """
    pooled = []
    for f in features:
        f = F.avg_pool2d(f, kernel_size=patch_size, stride=1, padding=patch_size // 2)
        pooled.append(f)
    return pooled


def combine_layers(pooled_features):
    """
    Aligne spatialement les feature maps de layer2 et layer3 (résolutions différentes)
    sur la résolution de la plus grande (layer2), puis les concatène sur les canaux.
    """
    target_size = pooled_features[0].shape[-2:]  # résolution de layer2 (plus grande)
    resized = [pooled_features[0]]

    for f in pooled_features[1:]:
        f_resized = F.interpolate(f, size=target_size, mode="bilinear", align_corners=False)
        resized.append(f_resized)

    combined = torch.cat(resized, dim=1)  # concat sur les canaux -> [B, C_total, H, W]
    return combined


def extract_patch_embeddings(backbone, images, patch_size=3):
    with torch.no_grad():
        features = backbone(images)          # [layer2_feat, layer3_feat]
        pooled = embed_patches(features, patch_size)
        combined = combine_layers(pooled)    # [B, C, H, W]

    B, C, H, W = combined.shape
    embeddings = combined.permute(0, 2, 3, 1).reshape(B, H * W, C)
    return embeddings, (B, H, W)

In [ ]:
from torch.utils.data import DataLoader

backbone = timm.create_model(
    "wide_resnet50_2", pretrained=True, features_only=True, out_indices=(2, 3)
).to(device).eval()

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

for batch in test_loader:
    images = batch["image"].to(device)
    labels = batch["label"]
    defect_types = batch["defect_type"]

    test_embeddings, (B, H, W) = extract_patch_embeddings(backbone, images)
    print(images.shape)
    print(test_embeddings.shape)
    break

torch.Size([16, 3, 256, 256])
torch.Size([16, 1024, 1536])


In [15]:
for b in test_loader:
    print(b['image'])
    print(b['label'])
    print(b["defect_type"])
    print(b['path'])
    break

tensor([[[[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]]],


        [[[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
        

In [ ]:
def load_embeddings(path):
    return torch.load(path, map_location="cpu")

data = load_embeddings("/home/2017025/ayouce01/FLOCAT-GENAGN/What-if-Flocat-were-truly-generic-and-agnostic/data/embeddings/wide_resnet50_2/bottle/train.pt")

embeddings = data["embeddings"]        
labels = data["labels"]                
defect_types = data["defect_types"]    
paths = data["paths"]                  
grid_hw = data["grid_hw"]              

/tmp/ipykernel_2597617/1577216172.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(path, map_location="cpu")


In [21]:
embeddings.shape

torch.Size([209, 1024, 1536])

In [25]:
grid_hw

(32, 32)